In [ ]:
import muon as mu
import numpy as np
import pandas as pd
import anndata
from scipy.sparse import csr_matrix

# Noise (subset proteins, flip values) and redundancy

'''
(1) noise — per cell this randomly subset which proteins are "measured" (variable panel size, ~1–257 proteins) and flips ~10% of the remaining values. 
(2) redundancy — this generates K independent noisy paragraphs per cell, so each cell has multiple views that can be
  used as positive pairs. Output is a CSV (cell_texts_augmented_K{K}_flip{FLIP_PROB}.csv) with columns cell_id, version, subset_size, text.
'''

DATA_PATH = '/home/lemanic-g04/'
SEED = 42
K = 5                          # number of paragraphs per cell
FLIP_PROB = 0.10
MIN_PROTEINS = 1
MAX_PROTEINS = 257
SHUFFLE_ORDER = True           # also randomize protein order in the sentence

rng = np.random.default_rng(SEED)

# Load clean binarized data
mdata = mu.read(DATA_PATH + 'GSE194315_binarized_with_text_mu.h5mu')
protein = mdata.mod['protein']
X = protein.X.toarray() if hasattr(protein.X, 'toarray') else protein.X
X = X.astype(bool)
protein_names = protein.var_names.to_numpy()
n_cells, n_proteins = X.shape

def humanize(names):
  names = list(names)
  if len(names) == 0: return ""
  if len(names) == 1: return names[0]
  if len(names) == 2: return f"{names[0]} and {names[1]}"
  return ", ".join(names[:-1]) + f", and {names[-1]}"

def cell_to_text(row, mask):
  expressed     = protein_names[row & mask]
  not_expressed = protein_names[(~row) & mask]
  if SHUFFLE_ORDER:
      expressed     = rng.permutation(expressed)
      not_expressed = rng.permutation(not_expressed)
  if len(expressed) == 0 and len(not_expressed) == 0:
      return "No surface proteins were measured for this cell."
  if len(expressed) == 0:
      return (f"This cell does not express any of the measured surface proteins. "
              f"Specifically, it lacks expression of {humanize(not_expressed)}.")
  if len(not_expressed) == 0:
      return (f"This cell expresses all of the measured surface proteins. "
              f"Specifically, it expresses {humanize(expressed)}.")
  return (f"This cell expresses the following surface proteins: {humanize(expressed)}. "
          f"However, it does not express {humanize(not_expressed)}.")

# Generate K paragraphs per cell
records = []
for k in range(K):
  print(f"Generating version {k+1}/{K}")
  for i in range(n_cells):
      # Per-cell subset
      size = int(rng.integers(MIN_PROTEINS, MAX_PROTEINS + 1))
      keep_idx = rng.choice(n_proteins, size=size, replace=False)
      mask = np.zeros(n_proteins, dtype=bool)
      mask[keep_idx] = True

      # Per-cell flips
      flips = (rng.random(n_proteins) < FLIP_PROB) & mask
      row_noisy = X[i] ^ flips

      # Text
      text = cell_to_text(row_noisy, mask)

      records.append({
          'cell_id': protein.obs_names[i],
          'version': k,
          'subset_size': size,
          'text': text,
      })

df = pd.DataFrame(records)
print(f"Generated {len(df)} paragraphs ({n_cells} cells × {K} versions)")

# Save
out_csv = f'cell_texts_augmented_K{K}_flip{int(FLIP_PROB*100)}.csv'
df.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")

In [ ]:
# Noise (subset proteins) and redundancy (different paragraphs per cell based on orderings)

'''
  (1) subsetting — per cell this randomly subsets which proteins are "measured" (variable panel size, ~1–257 proteins).
      Values are NOT flipped — the biology is preserved, only the observable panel changes.
  (2) redundancy — this generates K independent subsetted paragraphs per cell, so each cell has multiple views that can be
      used as positive pairs. Output is a CSV (cell_texts_augmented_K{K}_subsetOnly.csv) with columns cell_id, version, subset_size, text.
  '''

  DATA_PATH = '/home/lemanic-g04/'
  SEED = 42
  K = 5                          # number of paragraphs per cell
  MIN_PROTEINS = 1
  MAX_PROTEINS = 257
  SHUFFLE_ORDER = True           # also randomize protein order in the sentence

  rng = np.random.default_rng(SEED)

  # Load clean binarized data
  mdata = mu.read(DATA_PATH + 'GSE194315_binarized_with_text_mu.h5mu')
  protein = mdata.mod['protein']
  X = protein.X.toarray() if hasattr(protein.X, 'toarray') else protein.X
  X = X.astype(bool)
  protein_names = protein.var_names.to_numpy()
  n_cells, n_proteins = X.shape

  def humanize(names):
      names = list(names)
      if len(names) == 0: return ""
      if len(names) == 1: return names[0]
      if len(names) == 2: return f"{names[0]} and {names[1]}"
      return ", ".join(names[:-1]) + f", and {names[-1]}"

  def cell_to_text(row, mask):
      expressed     = protein_names[row & mask]
      not_expressed = protein_names[(~row) & mask]
      if SHUFFLE_ORDER:
          expressed     = rng.permutation(expressed)
          not_expressed = rng.permutation(not_expressed)
      if len(expressed) == 0 and len(not_expressed) == 0:
          return "No surface proteins were measured for this cell."
      if len(expressed) == 0:
          return (f"This cell does not express any of the measured surface proteins. "
                  f"Specifically, it lacks expression of {humanize(not_expressed)}.")
      if len(not_expressed) == 0:
          return (f"This cell expresses all of the measured surface proteins. "
                  f"Specifically, it expresses {humanize(expressed)}.")
      return (f"This cell expresses the following surface proteins: {humanize(expressed)}. "
              f"However, it does not express {humanize(not_expressed)}.")

  # Generate K paragraphs per cell
  records = []
  for k in range(K):
      print(f"Generating version {k+1}/{K}")
      for i in range(n_cells):
          # Per-cell subset
          size = int(rng.integers(MIN_PROTEINS, MAX_PROTEINS + 1))
          keep_idx = rng.choice(n_proteins, size=size, replace=False)
          mask = np.zeros(n_proteins, dtype=bool)
          mask[keep_idx] = True

          # No flips — use the clean values at measured positions
          row_clean = X[i]

          # Text
          text = cell_to_text(row_clean, mask)

          records.append({
              'cell_id': protein.obs_names[i],
              'version': k,
              'subset_size': size,
              'text': text,
          })

  df = pd.DataFrame(records)
  print(f"Generated {len(df)} paragraphs ({n_cells} cells × {K} versions)")

  # Save
  out_csv = f'cell_texts_augmented_K{K}_subsetOnly.csv'
  df.to_csv(out_csv, index=False)
  print(f"Saved {out_csv}")